# Introduction to Conditional Generative Adversarial Networks






## Getting Pytorch Going

Let's get started by importing pytorch and some of the more important sub-modules.



In [7]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np

import torch
import torch.nn as nn
from torchvision.datasets import ImageFolder, MNIST
from torchvision import transforms
from torchvision.transforms import functional as transformsFunctional
from torch import autograd
from torch.autograd import Variable
from torchvision.utils import make_grid

from torch.utils import data as t_data
import torchvision.datasets as datasets

from torch.utils.tensorboard import SummaryWriter

You will likely want to make sure this notebook has access to a GPU. Run the following cell.

In [2]:
gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
if gpu_info.find('failed') >= 0:
  print('Not connected to a GPU')
else:
  print(gpu_info)


torch.cuda.is_available()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

Mon Dec 11 22:44:18 2023       
+-----------------------------------------------------------------------------+
| NVIDIA-SMI 525.105.17   Driver Version: 525.105.17   CUDA Version: 12.0     |
|-------------------------------+----------------------+----------------------+
| GPU  Name        Persistence-M| Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp  Perf  Pwr:Usage/Cap|         Memory-Usage | GPU-Util  Compute M. |
|                               |                      |               MIG M. |
|===============================+======================+======================|
|   0  Tesla T4            Off  | 00000000:00:04.0 Off |                    0 |
| N/A   55C    P8    10W /  70W |      0MiB / 15360MiB |      0%      Default |
|                               |                      |                  N/A |
+-------------------------------+----------------------+----------------------+
                                                                               
+-------

# Load Fashion-MNIST dataset

In [3]:
batch_size = 32

transform = transforms.Compose([transforms.ToTensor(),
                                transforms.Normalize(mean=[0.5],
                                std=[0.5])]
)
fashionmnist_trainset = datasets.FashionMNIST(root='../data/', train=True, transform=transform, download=True)
data_loader = t_data.DataLoader(dataset=fashionmnist_trainset, batch_size=batch_size, shuffle=True, drop_last=True)

100%|██████████| 26421880/26421880 [00:02<00:00, 13191137.43it/s]


Extracting ../data/FashionMNIST/raw/train-images-idx3-ubyte.gz to ../data/FashionMNIST/raw



100%|██████████| 29515/29515 [00:00<00:00, 213943.33it/s]


Extracting ../data/FashionMNIST/raw/train-labels-idx1-ubyte.gz to ../data/FashionMNIST/raw



100%|██████████| 4422102/4422102 [00:01<00:00, 3887346.93it/s]


Extracting ../data/FashionMNIST/raw/t10k-images-idx3-ubyte.gz to ../data/FashionMNIST/raw



100%|██████████| 5148/5148 [00:00<00:00, 6078906.81it/s]

Extracting ../data/FashionMNIST/raw/t10k-labels-idx1-ubyte.gz to ../data/FashionMNIST/raw



Clase para definir un Discriminador simple, arquitectura MLP con cuatro capas y **un embedding para recibir la etiqueta** (principal diferencia con GAN).

La función *forward* recibe tanto las imágenes ***x*** como las etiquetas ***labels***. Las etiquetas pasan por el embedding y la salida se concatena con las imágenes.


In [4]:
class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()

        self.label_emb = nn.Embedding(10, 10)

        self.model = nn.Sequential(
            nn.Linear(794, 1024),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Dropout(0.3),
            nn.Linear(1024, 512),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Dropout(0.3),
            nn.Linear(256, 1),
            nn.Sigmoid()
        )

    def forward(self, x, labels):
        x = x.view(x.size(0), 784)
        c = self.label_emb(labels)
        x = torch.cat([x, c], 1)
        out = self.model(x)
        return out.squeeze()

Clase para definir un Generador simple, arquitectura MLP con cuatro capas y **un embedding para recibir la etiqueta** (principal diferencia con GAN).

La función *forward* recibe tanto los vectores del espacio latente ***z*** como las etiquetas ***labels***. Las etiquetas pasan por el embedding y la salida se concatena con las los vectores del espacio latente.

In [5]:
class Generator(nn.Module):
    def __init__(self):
        super().__init__()

        self.label_emb = nn.Embedding(10, 10)

        self.model = nn.Sequential(
            nn.Linear(110, 256),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(256, 512),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(512, 1024),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(1024, 784),
            nn.Tanh()
        )

    def forward(self, z, labels):
        z = z.view(z.size(0), 100)
        c = self.label_emb(labels)
        x = torch.cat([z, c], 1)
        out = self.model(x)
        return out.view(x.size(0), 28, 28)

Instanciamos las ANNs

In [6]:
generator = Generator().to(device)
discriminator = Discriminator().to(device)

criterion = nn.BCELoss()
learning_rate=0.0001
d_optimizer = torch.optim.Adam(discriminator.parameters(), lr=learning_rate)
g_optimizer = torch.optim.Adam(generator.parameters(), lr=learning_rate)

KeyboardInterrupt: ignored

In [ ]:
writer = SummaryWriter()

Entrenamiento del generador

In [ ]:
def generator_train_step(batch_size, discriminator, generator, g_optimizer, criterion):
    g_optimizer.zero_grad()

    # 1 Create fake data
    z = Variable(torch.randn(batch_size, 100)).to(device)
    fake_labels = Variable(torch.LongTensor(np.random.randint(0, 10, batch_size))).to(device)
    fake_images = generator(z, fake_labels)

    # 2 Try to fool the discriminator with fake data
    validity = discriminator(fake_images, fake_labels)
    g_loss = criterion(validity, Variable(torch.ones(batch_size)).to(device))
    g_loss.backward()

    # 3 Optimize the generator weights
    g_optimizer.step()

    return g_loss.item()

Entrenamiento del discriminador: vemos que recibe tanto las imágenes reales como las etiquetas de las imágenes

In [ ]:
def discriminator_train_step(batch_size, discriminator, generator, d_optimizer, criterion, real_images, labels):
    d_optimizer.zero_grad()

    # 1 Train discriminator on real data
    real_validity = discriminator(real_images, labels)
    real_loss = criterion(real_validity, Variable(torch.ones(batch_size)).to(device))

    # 2 Train the discriminator on data produced by the generator
    z = Variable(torch.randn(batch_size, 100)).to(device)
    fake_labels = Variable(torch.LongTensor(np.random.randint(0, 10, batch_size))).to(device)
    fake_images = generator(z, fake_labels)
    fake_validity = discriminator(fake_images, fake_labels)
    fake_loss = criterion(fake_validity, Variable(torch.zeros(batch_size)).to(device))

    # 3 Get loss to train the discriminator from both losses
    d_loss = real_loss + fake_loss
    d_loss.backward()

    # 3 Optimize the discriminator weights
    d_optimizer.step()
    return d_loss.item()

In [ ]:
num_epochs = 3
display_step = 3
for epoch in range(num_epochs):
    print('Starting epoch {}...'.format(epoch), end=' ')
    generator.train()     # Generator prepared to be trained. It is needed if we apply generator.eval()
    discriminator.train() # Discriminator prepared to be trained.
    for i, (images, labels) in enumerate(data_loader):

        step = epoch * len(data_loader) + i + 1   # For the stats

        real_images = Variable(images).to(device) # Get real images
        labels = Variable(labels).to(device)      # Get real labels

        d_loss = discriminator_train_step(len(real_images), discriminator,
                                          generator, d_optimizer, criterion,
                                          real_images, labels)


        g_loss = generator_train_step(batch_size, discriminator, generator,
                                      g_optimizer, criterion)

        writer.add_scalars('scalars', {'g_loss': g_loss, 'd_loss': d_loss}, step)

        if step % display_step == 0:
            generator.eval()
            z = Variable(torch.randn(9, 100)).to(device)
            labels = Variable(torch.LongTensor(np.arange(9))).to(device)
            sample_images = generator(z, labels).unsqueeze(1)
            grid = make_grid(sample_images, nrow=3, normalize=True)
            writer.add_image('sample_image', grid, step)

    print('Epoch={}, Discriminator loss={}, Generator loss={}'.format(epoch, d_loss, g_loss))

    print('Done!')

In [ ]:
torch.save(generator.state_dict(), 'generator_state.pt')

In [ ]:
z = Variable(torch.randn(100, 100)).to(device)
labels = torch.LongTensor([i for i in range(10) for _ in range(10)]).to(device)

In [ ]:
images = generator(z, labels).unsqueeze(1)

In [ ]:
grid = make_grid(images.cpu(), nrow=10, normalize=True)

In [ ]:
fig, ax = plt.subplots(figsize=(10,10))
ax.imshow(grid.permute(1, 2, 0).data, cmap='binary')
ax.axis('off')

In [ ]:
def generate_clothing(generator, label):
    z = Variable(torch.randn(1, 100)).to(device)
    label = torch.LongTensor([label]).to(device)
    img = generator(z, label).data.cpu()
    img = 0.5 * img + 0.5
    return transforms.ToPILImage()(img)

In [ ]:
generate_clothing(generator, 0)